In [0]:

storage_account_name = "aerocloudstorage"
storage_account_key = "<MA_CLE_AZURE_SECRETE>"

# Configuration de Spark pour accéder à ADLS Gen2
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

# chemins d'accès
bronze_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/opensky/*/*/*/*.json"
silver_path = f"abfss://silver@{storage_account_name}.dfs.core.windows.net/flights/"

In [0]:
from pyspark.sql.functions import col, explode, current_timestamp

# lecture des fichiers JSON bruts
df_bronze = spark.read.json(bronze_path)
df_exploded = df_bronze.select(explode(col("states")).alias("state"))

# vraies colonnes métier
df_silver = df_exploded.select(
    col("state")[0].cast("string").alias("icao24"),
    col("state")[1].cast("string").alias("callsign"),
    col("state")[2].cast("string").alias("origin_country"),
    col("state")[5].cast("double").alias("longitude"),
    col("state")[6].cast("double").alias("latitude"),
    col("state")[7].cast("double").alias("altitude_baro"),
    col("state")[8].cast("boolean").alias("on_ground"),
    col("state")[9].cast("double").alias("velocity"),
    col("state")[10].cast("double").alias("true_track"),
    current_timestamp().alias("ingestion_timestamp") 
)

# Nettoyage 
df_silver_clean = df_silver.filter(col("longitude").isNotNull() & col("latitude").isNotNull())

display(df_silver_clean)

icao24,callsign,origin_country,longitude,latitude,altitude_baro,on_ground,velocity,true_track,ingestion_timestamp
39de4f,TVF97HM,France,6.0279,45.0589,11582.4,false,226.83,319.05,2026-03-03T12:42:24.61029Z
4b1819,SWR1CG,Switzerland,3.668,50.1803,11894.82,false,221.58,133.78,2026-03-03T12:42:24.61029Z
39de4a,TVF78LC,France,-1.5206,43.7748,11574.78,false,235.18,25.39,2026-03-03T12:42:24.61029Z
407a38,NPT905,United Kingdom,-2.0235,49.6631,4617.72,false,120.49,2.45,2026-03-03T12:42:24.61029Z
39de50,TVF67KL,France,7.6003,45.5389,11590.02,false,225.18,315.46,2026-03-03T12:42:24.61029Z
39de55,TVF29NG,France,2.3071,48.7164,106.68,false,66.51,61.85,2026-03-03T12:42:24.61029Z
39de54,TVF66YR,France,6.7583,46.1841,11582.4,false,230.23,311.92,2026-03-03T12:42:24.61029Z
4952c4,TAP556,Portugal,-1.1168,42.6567,10363.2,false,210.37,78.0,2026-03-03T12:42:24.61029Z
39de45,TVF44VH,France,-1.2055,46.3472,10972.8,false,217.58,43.85,2026-03-03T12:42:24.61029Z
39de48,TVF26NP,France,-0.7572,50.9106,10058.4,false,207.69,148.99,2026-03-03T12:42:24.61029Z


In [0]:
# sauvegarde dans le conteneur silver au format delta
df_silver_clean.write \
    .format("delta") \
    .mode("append") \
    .save(silver_path)

print(f"Données traitées et sauvegardées avec succès dans {silver_path}")

Données traitées et sauvegardées avec succès dans abfss://silver@aerocloudstorage.dfs.core.windows.net/flights/
